# Week 13, Lab 1 — Statistical evidence for a recovery difference

**HWRS 564a · Tuesday, November 17, 2026 · 75-minute lecture with embedded lab**

**Question:** Does the Pinal model reproduce the recovery rates in its historical head observations?

You will describe a distribution, estimate an average paired difference, and use simulation to investigate uncertainty. The four coding tasks are short; spend time on the interpretation prompts.

Run cells in order with **Shift+Enter**. Complete `# YOUR TURN` cells; `# CHECK` cells verify the numerical work. No downloads, model archive, or MODFLOW run are needed. Thursday's lab starts independently from the same committed data.

By the end you should distinguish variation among wells, uncertainty in an estimated mean, and a distribution generated under a null model. You should also state why nearby wells may not provide independent evidence.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works from the repository root, this lab folder, or its generated solution folder.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/cache/pinal_model_head_obs.csv").exists())
BLUE, RED = "#0C234B", "#AB0520"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})

## 1 — The comparison and the observational unit · minutes 0–22

Week 12 compared summaries across well populations. Here we use **matched observed and simulated heads at the same site and time** from the Pinal model's HOB cache.

The supplied table fits each site's observed and simulated heads during **1985 ≤ year < 2005**, using the same rows. A site must have at least five observations spanning ten years. These are head elevations, so positive slopes mean rising water levels.

For each site, define

$$d_i=b_{obs,i}-b_{sim,i}.$$

Positive $d_i$ means the simulation's recovery slope is smaller. Negative $d_i$ means its slope is larger. Some sites decline even within this recovery period.

**Target:** the equally weighted mean slope difference across the qualifying monitored sites. This is not an area-weighted basin average, nor a random sample of every well in Pinal County.

The summary is generated by `scripts/prepare_week13_data.py` from `data/cache/pinal_model_head_obs.csv`; Thursday we reproduce one row.

In [ ]:
paired = pd.read_csv(ROOT / "data/week13_pinal_recovery_slopes.csv", dtype={"site": str})
assert paired["site"].is_unique
assert len(paired) == 191
paired.head()

### YOUR TURN 1 — Describe the effect · minutes 22–27

Extract `gap_ftyr` as a NumPy array `gap`. Calculate `n_sites`, `mean_gap`, `median_gap`, and `sd_gap` (sample standard deviation: `ddof=1`). All summaries except the count have units ft/year.

**Before running:** does positive gap mean that the simulated head is always too low? Explain why a difference in *slopes* cannot establish an offset in *heads*.

In [ ]:
# YOUR TURN
gap = ...
n_sites = ...
mean_gap = ...
median_gap = ...
sd_gap = ...

In [ ]:
# CHECK
assert n_sites == 191
assert np.isclose(mean_gap, 1.4416016334, atol=1e-8)
assert np.isclose(median_gap, 0.9524022351, atol=1e-8)
assert np.isclose(sd_gap, np.std(paired["gap_ftyr"], ddof=1))
print(f"{n_sites} sites: mean {mean_gap:+.2f}, median {median_gap:+.2f}, SD {sd_gap:.2f} ft/year")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(gap, bins=25, color=BLUE, alpha=0.8)
ax.axvline(0, color="0.4", lw=1, label="equal slopes")
ax.axvline(mean_gap, color=RED, ls="--", lw=2, label=f"mean {mean_gap:+.2f} ft/year")
ax.set(xlabel="observed minus simulated recovery slope (ft/year)", ylabel="sites",
       title="Recovery differences vary substantially among monitored sites")
ax.legend(frameon=False)
plt.show()

### YOUR TURN 2 — Which sites receive weight? · minutes 27–32

Each row in `paired` represents a site, but `n` records how many head observations supported its slopes. Calculate `row_weighted_gap` with `np.average(gap, weights=paired["n"])` and `largest_weight` as the largest site's fraction of the total `n`.

This weighted statistic is the average site slope difference **weighted by observation count**. It is not the slope from pooling all head records. Compare it with the equal-site mean. Which question gives densely measured sites more influence?

In [ ]:
# YOUR TURN
row_weighted_gap = ...
largest_weight = ...

In [ ]:
# CHECK
assert np.isclose(row_weighted_gap, np.average(gap, weights=paired["n"]))
assert np.isclose(largest_weight, paired["n"].max() / paired["n"].sum())
print(f"Equal-site mean: {mean_gap:+.2f} ft/year")
print(f"Observation-count weighted mean: {row_weighted_gap:+.2f} ft/year")
print(f"Largest site's share of observations: {largest_weight:.1%}")

## 2 — Uncertainty in the average · minutes 32–50

The spread among sites is not the uncertainty in their mean. A **bootstrap** approximates the sampling variation of an estimator by resampling the observed units with replacement.

Imagine the monitored sites as an independent sample from a broader population of comparable sites. Draw 191 site differences with replacement, calculate their mean, and repeat. A site can appear several times in one draw, and another can be absent.

The mean for these exact 191 recorded slopes is already known. The interval concerns generalizing beyond them under a sampling model. It does not repair preferential monitoring locations or include every source of uncertainty.

### YOUR TURN 3 — Bootstrap the mean

Use the supplied seeded generator. Fill `boot_means` with **5,000** resampled means, then calculate `ci_low, ci_high` as the 2.5th and 97.5th percentiles. `rng.choice(gap, size=len(gap), replace=True)` draws one replicate.

In [ ]:
rng = np.random.default_rng(56413)
B = 5000
# YOUR TURN
boot_means = ...
ci_low, ci_high = ...

In [ ]:
# CHECK
check_rng = np.random.default_rng(56413)
expected = np.array([check_rng.choice(gap, size=len(gap), replace=True).mean() for _ in range(B)])
assert np.allclose(boot_means, expected)
assert np.allclose([ci_low, ci_high], np.quantile(expected, [0.025, 0.975]))
assert ci_low < mean_gap < ci_high
print(f"Illustrative independent-site bootstrap interval: [{ci_low:+.2f}, {ci_high:+.2f}] ft/year")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(gap, bins=25, color=BLUE)
axes[0].set(xlabel="site slope difference (ft/year)", ylabel="sites", title="Spread among sites")
axes[1].hist(boot_means, bins=35, color=BLUE)
axes[1].axvline(ci_low, color=RED, ls="--")
axes[1].axvline(ci_high, color=RED, ls="--", label="95% percentile interval")
axes[1].set(xlabel="resampled mean difference (ft/year)", ylabel="bootstrap replicates",
            title="Variation of the estimated mean")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

**Discuss:** Why is the distribution on the right narrower? Its horizontal axis has the same units, but its values represent a different quantity.

A 95% confidence procedure aims to cover the population quantity in about 95% of repeated samples when its assumptions hold. This bootstrap interval is approximate. It is not a range containing 95% of sites, and it is not a 95% probability statement about a fixed parameter after seeing the data.

**Assumption checkpoint:** nearby sites may respond to the same stresses. Independent-site resampling ignores that spatial dependence. It also treats the fitted site slopes as the inputs without propagating their individual estimation errors. Treat this interval as an instructional calculation, not a complete uncertainty analysis of the Pinal model.

## 3 — A specified null model · minutes 50–65

A different question is: *Would a mean difference this far from zero be unusual under a specified no-systematic-difference model?*

For this demonstration assume independent site differences with distributions symmetric around zero. Conditional on each magnitude, positive and negative signs are equally plausible. Randomly reverse signs and recalculate the mean. This symmetry is stronger than merely asserting a zero mean; it is not guaranteed by the Pinal data.

This is a model-based demonstration, not a randomized experiment. Its p-value is conditional on the null assumptions. Dependence and asymmetry can make its nominal interpretation unreliable.

In [ ]:
null_rng = np.random.default_rng(130564)
null_means = np.array([
    (gap * null_rng.choice([-1, 1], size=len(gap))).mean()
    for _ in range(B)
])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(null_means, bins=35, color="0.65")
ax.axvline(mean_gap, color=RED, ls="--", label="observed mean")
ax.axvline(-abs(mean_gap), color=RED, ls=":", label="equally extreme opposite tail")
ax.set(xlabel="mean under sign-flip null model (ft/year)", ylabel="simulations")
ax.legend(frameon=False)
plt.show()

### YOUR TURN 4 — Read the null distribution

Count simulated means whose absolute values are **at least** the absolute observed mean. Store the count in `n_extreme`, then calculate the Monte Carlo p-value as `(n_extreme + 1) / (B + 1)`.

The correction prevents reporting a probability of zero when a finite simulation produces no exceedances. It does not fix a poorly specified null model.

In [ ]:
# YOUR TURN
n_extreme = ...
p_value = ...

In [ ]:
# CHECK
assert n_extreme == int(np.count_nonzero(np.abs(null_means) >= abs(mean_gap)))
assert np.isclose(p_value, (n_extreme + 1) / (B + 1))
assert 0 < p_value <= 1
print(f"{n_extreme} of {B} null simulations at least as extreme; Monte Carlo p = {p_value:.5f}")

A p-value describes how often the specified null model produces a statistic at least as extreme as ours. It is **not** the probability the null is true, a measure of the size of the difference, or proof of a cause.

A non-small p-value would not establish agreement. Small effects can be detectable with abundant independent information; important effects can be difficult to detect with sparse or dependent observations.

## 4 — The claim the evidence supports · minutes 65–75

Write four sentences:

1. State the mean paired slope difference, its sign, units, period, and the sites it describes.
2. Give the illustrative interval and state the independent-site assumption.
3. Explain the null-model result without assigning a probability to the null hypothesis.
4. Name a limitation that matters for using the model to predict future responses to reduced pumping.

**Your conclusion:**

*Write here.*

**Partner check:** Does the conclusion accidentally generalize to every well or claim that a particular pumping or recharge input caused the discrepancy?

Before leaving: restart the kernel, run all cells, and save. Thursday examines the slopes and model errors behind this summary.

## Sources and preparation

Data: the committed Pinal HOB cache used in Week 12. Rebuild the slope table with `python scripts/prepare_week13_data.py` from the repository root.

Statistical sequence informed by *Data Analysis in Water Sciences*, Steven Pestana, Jessica D. Lundquist, and contributors: [sampling and distributions](https://mountain-hydrology-research-group.github.io/data-analysis/modules/module1/lab1-2.html) and [Monte Carlo methods](https://mountain-hydrology-research-group.github.io/data-analysis/modules/module2/lab2-4.html). This notebook uses the course's Pinal data and a new paired comparison; the independent-site bootstrap is an instructional adaptation.